# 05 — Baseline Inference (Variants A, B, C)

Run the open-source baselines using **Qwen3-4B-Instruct** (latest Qwen generation, closest
dense model to 3B — Qwen3 has no 3B dense variant).

**Requires GPU** — or CPU with patience for a few examples.

**Variants covered:**

| Variant | Description | Model | Decoding |
|---------|-------------|-------|----------|
| A | Zero-shot, greedy | Qwen3-4B-Instruct-2507 | Unconstrained |
| B | Zero-shot + constrained JSON | Qwen3-4B-Instruct-2507 | JSON Schema via `outlines` |
| C | Few-shot (3 examples), greedy | Qwen3-4B-Instruct-2507 | Unconstrained |

Variant E (proprietary API) is left for a separate run since it requires API keys.

**What we do:**

1. Load model and tokenizer
2. Load data splits, schemas, and prompt templates from previous notebooks
3. Run Variant A on a small sample → inspect outputs, iterate if needed
4. Run Variant B with constrained decoding → compare structural correctness
5. Run Variant C with few-shot examples → compare extraction quality
6. Run all variants on the full val set → save predictions
7. Log latency and token counts per example

In [ ]:
import json
import time
import re
from pathlib import Path
from collections import Counter, defaultdict

import torch
import pandas as pd
from jinja2 import Template
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

CLEANED_DIR = Path("../data/WDC-PAVE/cleaned")
PRED_DIR = Path("../data/WDC-PAVE/predictions")
PRED_DIR.mkdir(exist_ok=True)

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

# Device setup
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")
print(f"Model:  {MODEL_ID}")

## 1. Load model and tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Use bfloat16 for efficiency; device_map="auto" dispatches across available devices
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"dtype: {model.dtype}")
print(f"Device map: {model.hf_device_map if hasattr(model, 'hf_device_map') else device}")

## 2. Load data and prompt templates

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f]

train = load_jsonl(CLEANED_DIR / "train.jsonl")
val = load_jsonl(CLEANED_DIR / "val.jsonl")
test = load_jsonl(CLEANED_DIR / "test.jsonl")

with open(CLEANED_DIR / "category_schemas.json") as f:
    CATEGORY_SCHEMAS = json.load(f)

with open(CLEANED_DIR / "json_schemas_for_decoding.json") as f:
    JSON_SCHEMAS = json.load(f)

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

# Index train by category for few-shot selection
train_by_cat: dict[str, list[dict]] = defaultdict(list)
for rec in train:
    train_by_cat[rec["category"]].append(rec)

In [ ]:
# ---- Prompt templates (same as notebook 04) ----

SYSTEM_MESSAGE = """\
You are an information extraction system. Extract product attributes from the given title and description.

Rules:
- Return exactly one valid JSON object.
- Use exactly the attribute names listed by the user as keys.
- If an attribute is missing or cannot be determined, return null.
- If multiple values apply, return them as a JSON array sorted alphabetically.
- Do not add extra keys. Do not explain. Use only information from the input text.
- The word "Null" in the input is a data artifact — ignore it."""

USER_TEMPLATE = Template("""\
Category: {{ category }}

Attributes:
{{ attribute_list }}

Title: {{ title }}

Description: {{ description }}""")


def build_messages_zero_shot(record: dict, schemas: dict[str, list[str]]) -> list[dict]:
    """Build chat messages for zero-shot inference (Variants A, B)."""
    category = record["category"]
    attrs = schemas[category]
    user_msg = USER_TEMPLATE.render(
        category=category,
        attribute_list=", ".join(attrs),
        title=record["input_title"],
        description=record["input_description"],
    )
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
    ]


def build_messages_few_shot(
    record: dict,
    schemas: dict[str, list[str]],
    train_by_cat: dict[str, list[dict]],
    n_examples: int = 3,
) -> list[dict]:
    """Build chat messages with few-shot examples (Variant C).
    
    Examples are added as user/assistant turn pairs before the target record.
    """
    category = record["category"]
    attrs = schemas[category]
    examples = select_few_shot_examples(category, train_by_cat, n=n_examples, exclude_id=record["id"])
    
    messages = [{"role": "system", "content": SYSTEM_MESSAGE}]
    
    for ex in examples:
        ex_user = USER_TEMPLATE.render(
            category=category,
            attribute_list=", ".join(attrs),
            title=ex["input_title"],
            description=ex["input_description"],
        )
        messages.append({"role": "user", "content": ex_user})
        messages.append({"role": "assistant", "content": json.dumps(ex["gold_json"])})
    
    # The actual target record
    target_user = USER_TEMPLATE.render(
        category=category,
        attribute_list=", ".join(attrs),
        title=record["input_title"],
        description=record["input_description"],
    )
    messages.append({"role": "user", "content": target_user})
    return messages


def compute_fill_ratio(record: dict) -> float:
    gold = record["gold_json"]
    filled = sum(1 for v in gold.values() if v is not None)
    return filled / len(gold)


def select_few_shot_examples(
    category: str,
    train_by_cat: dict[str, list[dict]],
    n: int = 3,
    exclude_id: int | None = None,
) -> list[dict]:
    """Same-category, diversity-sampled (evenly spaced by fill ratio)."""
    pool = [r for r in train_by_cat[category] if r["id"] != exclude_id]
    pool.sort(key=compute_fill_ratio)
    if len(pool) <= n:
        return pool
    step = len(pool) / n
    indices = [int(i * step + step / 2) for i in range(n)]
    return [pool[i] for i in indices]


print("Prompt builders defined.")

## 3. Inference helpers

Core generation function: takes chat messages, tokenizes with `apply_chat_template`,
runs `model.generate()`, decodes only the new tokens, and extracts the JSON from the response.

Qwen3 has a "thinking" mode by default. We disable it with `enable_thinking=False` in
`apply_chat_template` to get direct JSON output without `<think>...</think>` blocks.

In [ ]:
@torch.no_grad()
def generate_response(
    messages: list[dict],
    max_new_tokens: int = 512,
    temperature: float | None = None,
) -> dict:
    """Generate a response from the model given chat messages.
    
    Returns a dict with:
      - raw_output: the full decoded text from the model
      - parsed_json: the parsed JSON dict (or None if parsing failed)
      - parse_error: error message if JSON parsing failed
      - input_tokens: number of input tokens
      - output_tokens: number of output tokens
      - latency_s: generation time in seconds
    """
    # Tokenize with chat template — disable Qwen3 thinking mode for direct output
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(
        [text],
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)
    
    input_len = inputs["input_ids"].shape[1]
    
    # Generate — greedy (temperature=None or 0 means do_sample=False)
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
    )
    if temperature is not None and temperature > 0:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = 0.95
    else:
        gen_kwargs["do_sample"] = False
    
    t0 = time.perf_counter()
    output_ids = model.generate(**inputs, **gen_kwargs)
    latency = time.perf_counter() - t0
    
    # Decode only the new tokens
    new_tokens = output_ids[0, input_len:]
    output_tokens = len(new_tokens)
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    # Try to parse JSON from the output
    parsed_json = None
    parse_error = None
    try:
        parsed_json = json.loads(raw_output)
    except json.JSONDecodeError:
        # Try to extract JSON from the output (model might add text around it)
        json_match = re.search(r"\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}", raw_output, re.DOTALL)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
            except json.JSONDecodeError as e:
                parse_error = f"JSON extraction failed: {e}"
        else:
            parse_error = f"No JSON object found in output"
    
    return {
        "raw_output": raw_output,
        "parsed_json": parsed_json,
        "parse_error": parse_error,
        "input_tokens": input_len,
        "output_tokens": output_tokens,
        "latency_s": round(latency, 3),
    }


print("generate_response() defined.")

## 4. Variant A — Zero-shot smoke test

Run on a handful of records first to verify the pipeline works and inspect outputs before
committing to the full val set.

In [ ]:
# Pick one val record per category for a quick smoke test
smoke_records = []
for cat in CATEGORY_SCHEMAS:
    rec = next(r for r in val if r["category"] == cat)
    smoke_records.append(rec)

print(f"Smoke test: {len(smoke_records)} records (one per category)\n")

smoke_results = []
for rec in smoke_records:
    messages = build_messages_zero_shot(rec, CATEGORY_SCHEMAS)
    result = generate_response(messages)
    smoke_results.append((rec, result))
    
    print(f"=== {rec['category']} (ID={rec['id']}) ===")
    print(f"  Latency: {result['latency_s']}s | In: {result['input_tokens']} tok | Out: {result['output_tokens']} tok")
    if result["parse_error"]:
        print(f"  PARSE ERROR: {result['parse_error']}")
        print(f"  Raw output: {result['raw_output'][:300]}")
    else:
        print(f"  Parsed JSON keys: {list(result['parsed_json'].keys())}")
    print()

In [ ]:
# Compare predicted vs gold for the smoke test records
for rec, result in smoke_results:
    print(f"=== {rec['category']} (ID={rec['id']}) ===")
    gold = rec["gold_json"]
    pred = result["parsed_json"]
    
    if pred is None:
        print("  No parsed JSON — skipping comparison")
        print(f"  Raw: {result['raw_output'][:200]}")
        print()
        continue
    
    schema = CATEGORY_SCHEMAS[rec["category"]]
    print(f"  {'Attribute':<30s} {'Gold':<40s} {'Predicted':<40s} {'Match'}")
    print(f"  {'-'*30} {'-'*40} {'-'*40} {'-'*5}")
    for attr in schema:
        g = gold.get(attr)
        p = pred.get(attr)
        g_str = json.dumps(g) if g is not None else "null"
        p_str = json.dumps(p) if p is not None else "null"
        match = "OK" if g == p else "---"
        # Case-insensitive check
        if match == "---" and isinstance(g, str) and isinstance(p, str) and g.lower() == p.lower():
            match = "~ci"
        print(f"  {attr:<30s} {g_str:<40s} {p_str:<40s} {match}")
    print()

## 5. Full inference runner

Now we define a reusable runner that processes a list of records and collects predictions
in a standardized format for evaluation.

In [ ]:
def run_inference(
    records: list[dict],
    message_builder,
    variant_name: str,
    max_new_tokens: int = 512,
    temperature: float | None = None,
) -> list[dict]:
    """Run inference on a list of records and return standardized prediction dicts.
    
    message_builder: callable(record) -> list[dict] of chat messages
    """
    predictions = []
    
    for rec in tqdm(records, desc=f"Variant {variant_name}"):
        messages = message_builder(rec)
        result = generate_response(messages, max_new_tokens=max_new_tokens, temperature=temperature)
        
        predictions.append({
            "id": rec["id"],
            "category": rec["category"],
            "variant": variant_name,
            "model": MODEL_ID,
            "gold_json": rec["gold_json"],
            "pred_json": result["parsed_json"],
            "raw_output": result["raw_output"],
            "parse_error": result["parse_error"],
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "latency_s": result["latency_s"],
        })
    
    # Quick summary
    n_parsed = sum(1 for p in predictions if p["pred_json"] is not None)
    avg_latency = sum(p["latency_s"] for p in predictions) / len(predictions)
    print(f"\n{variant_name}: {n_parsed}/{len(predictions)} parsed "
          f"({n_parsed/len(predictions)*100:.1f}%), avg latency {avg_latency:.2f}s")
    
    return predictions


def save_predictions(predictions: list[dict], path: Path):
    """Save predictions as JSONL."""
    with open(path, "w") as f:
        for pred in predictions:
            f.write(json.dumps(pred) + "\n")
    print(f"Saved {len(predictions)} predictions to {path}")


print("run_inference() and save_predictions() defined.")

## 6. Variant A — Zero-shot on full val set

In [ ]:
preds_a = run_inference(
    records=val,
    message_builder=lambda rec: build_messages_zero_shot(rec, CATEGORY_SCHEMAS),
    variant_name="A_zero_shot",
)
save_predictions(preds_a, PRED_DIR / "variant_A_zero_shot_val.jsonl")

## 7. Variant B — Zero-shot + constrained JSON decoding

Uses `outlines` to constrain the model's output to valid JSON matching the category schema.
This isolates whether structural errors (invalid JSON, wrong keys) are a significant problem.

In [ ]:
import outlines

# Build an outlines model wrapper around the already-loaded transformers model
outlines_model = outlines.from_transformers(model, tokenizer)

print(f"Outlines model ready (wrapping {MODEL_ID})")

In [ ]:
# Build outlines JSON generators — one per category schema
json_generators = {}
for cat, schema in JSON_SCHEMAS.items():
    json_generators[cat] = outlines.generate.json(outlines_model, schema)
    print(f"  Built JSON generator for: {cat}")

print(f"\n{len(json_generators)} generators ready.")

In [ ]:
def run_constrained_inference(records: list[dict]) -> list[dict]:
    """Run Variant B: zero-shot with outlines constrained JSON decoding."""
    predictions = []
    
    for rec in tqdm(records, desc="Variant B (constrained)"):
        messages = build_messages_zero_shot(rec, CATEGORY_SCHEMAS)
        
        # Format the prompt the same way as unconstrained
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        
        category = rec["category"]
        generator = json_generators[category]
        
        t0 = time.perf_counter()
        try:
            result = generator(text, max_tokens=512)
            latency = time.perf_counter() - t0
            
            # outlines returns the parsed object directly
            if isinstance(result, dict):
                pred_json = result
            elif isinstance(result, str):
                pred_json = json.loads(result)
            else:
                # Pydantic model or similar — convert to dict
                pred_json = dict(result) if hasattr(result, "__dict__") else json.loads(str(result))
            
            parse_error = None
            raw_output = json.dumps(pred_json)
        except Exception as e:
            latency = time.perf_counter() - t0
            pred_json = None
            parse_error = f"Constrained generation failed: {e}"
            raw_output = str(e)
        
        predictions.append({
            "id": rec["id"],
            "category": category,
            "variant": "B_constrained",
            "model": MODEL_ID,
            "gold_json": rec["gold_json"],
            "pred_json": pred_json,
            "raw_output": raw_output,
            "parse_error": parse_error,
            "input_tokens": -1,  # outlines doesn't easily expose this
            "output_tokens": -1,
            "latency_s": round(latency, 3),
        })
    
    n_parsed = sum(1 for p in predictions if p["pred_json"] is not None)
    avg_latency = sum(p["latency_s"] for p in predictions) / len(predictions)
    print(f"\nB_constrained: {n_parsed}/{len(predictions)} parsed "
          f"({n_parsed/len(predictions)*100:.1f}%), avg latency {avg_latency:.2f}s")
    
    return predictions

print("run_constrained_inference() defined.")

In [ ]:
preds_b = run_constrained_inference(val)
save_predictions(preds_b, PRED_DIR / "variant_B_constrained_val.jsonl")

## 8. Variant C — Few-shot (3 examples) on full val set

In [ ]:
preds_c = run_inference(
    records=val,
    message_builder=lambda rec: build_messages_few_shot(rec, CATEGORY_SCHEMAS, train_by_cat, n_examples=3),
    variant_name="C_few_shot_3",
)
save_predictions(preds_c, PRED_DIR / "variant_C_few_shot_3_val.jsonl")

## 9. Quick comparison across variants

Before the full evaluation (notebook 08), let's get a rough sense of how the variants compare
on basic metrics: JSON parse rate, schema key match, and a few example-level comparisons.

In [ ]:
def quick_metrics(predictions: list[dict]) -> dict:
    """Compute quick sanity metrics for a set of predictions."""
    n = len(predictions)
    n_parsed = sum(1 for p in predictions if p["pred_json"] is not None)
    
    # Schema key match: does the predicted JSON have exactly the right keys?
    n_schema_ok = 0
    for p in predictions:
        if p["pred_json"] is not None:
            expected_keys = set(CATEGORY_SCHEMAS[p["category"]])
            actual_keys = set(p["pred_json"].keys())
            if expected_keys == actual_keys:
                n_schema_ok += 1
    
    # Exact match: prediction == gold (strict)
    n_exact = sum(1 for p in predictions
                  if p["pred_json"] is not None and p["pred_json"] == p["gold_json"])
    
    # Latency
    latencies = [p["latency_s"] for p in predictions]
    
    return {
        "n": n,
        "valid_json_pct": round(n_parsed / n * 100, 1),
        "schema_match_pct": round(n_schema_ok / n * 100, 1),
        "exact_match_pct": round(n_exact / n * 100, 1),
        "avg_latency_s": round(sum(latencies) / n, 2),
        "total_time_s": round(sum(latencies), 1),
    }


# Compare all variants
all_preds = {
    "A — Zero-shot": preds_a,
    "B — Constrained": preds_b,
    "C — Few-shot (3)": preds_c,
}

comparison_rows = []
for name, preds in all_preds.items():
    metrics = quick_metrics(preds)
    metrics["variant"] = name
    comparison_rows.append(metrics)

comp_df = pd.DataFrame(comparison_rows).set_index("variant")
print("Quick comparison across variants (val set):\n")
print(comp_df.to_string())

In [ ]:
# Per-category breakdown for each variant
for name, preds in all_preds.items():
    print(f"\n=== {name} — per-category valid JSON rate ===")
    cat_stats = defaultdict(lambda: {"total": 0, "parsed": 0, "schema_ok": 0})
    for p in preds:
        cat_stats[p["category"]]["total"] += 1
        if p["pred_json"] is not None:
            cat_stats[p["category"]]["parsed"] += 1
            expected = set(CATEGORY_SCHEMAS[p["category"]])
            if set(p["pred_json"].keys()) == expected:
                cat_stats[p["category"]]["schema_ok"] += 1
    
    for cat in CATEGORY_SCHEMAS:
        s = cat_stats[cat]
        parse_pct = s["parsed"] / s["total"] * 100 if s["total"] else 0
        schema_pct = s["schema_ok"] / s["total"] * 100 if s["total"] else 0
        print(f"  {cat:35s}  valid_json={parse_pct:5.1f}%  schema_match={schema_pct:5.1f}%  (n={s['total']})")

## 10. Inspect failure cases

Look at records where parsing failed or the schema didn't match — this tells us what
the model is actually doing wrong and informs prompt iteration.

In [ ]:
# Show parse failures from Variant A (most common failure mode for unconstrained decoding)
parse_failures = [p for p in preds_a if p["pred_json"] is None]
print(f"Variant A parse failures: {len(parse_failures)} / {len(preds_a)}\n")

for p in parse_failures[:5]:
    print(f"  ID={p['id']} ({p['category']})")
    print(f"  Error: {p['parse_error']}")
    print(f"  Raw output (first 300 chars):")
    print(f"    {p['raw_output'][:300]}")
    print()

In [ ]:
# Show schema mismatches from Variant A (parsed but wrong keys)
schema_mismatches = []
for p in preds_a:
    if p["pred_json"] is not None:
        expected = set(CATEGORY_SCHEMAS[p["category"]])
        actual = set(p["pred_json"].keys())
        if expected != actual:
            schema_mismatches.append({
                **p,
                "missing_keys": expected - actual,
                "extra_keys": actual - expected,
            })

print(f"Variant A schema mismatches: {len(schema_mismatches)} / {len(preds_a)}\n")

for p in schema_mismatches[:5]:
    print(f"  ID={p['id']} ({p['category']})")
    if p["missing_keys"]:
        print(f"    Missing: {p['missing_keys']}")
    if p["extra_keys"]:
        print(f"    Extra:   {p['extra_keys']}")
    print()

## 11. Latency and token usage summary

In [ ]:
latency_rows = []
for name, preds in all_preds.items():
    for p in preds:
        latency_rows.append({
            "variant": name,
            "category": p["category"],
            "latency_s": p["latency_s"],
            "input_tokens": p["input_tokens"],
            "output_tokens": p["output_tokens"],
        })

lat_df = pd.DataFrame(latency_rows)

print("Latency statistics per variant:\n")
print(lat_df.groupby("variant")["latency_s"].describe().round(2))
print()

# Token usage (Variant A and C — B doesn't track tokens)
for name in ["A — Zero-shot", "C — Few-shot (3)"]:
    subset = lat_df[lat_df["variant"] == name]
    print(f"{name}:")
    print(f"  Input tokens:  mean={subset['input_tokens'].mean():.0f}, "
          f"max={subset['input_tokens'].max()}")
    print(f"  Output tokens: mean={subset['output_tokens'].mean():.0f}, "
          f"max={subset['output_tokens'].max()}")
    print()

## 12. Save all outputs and list prediction files

In [ ]:
# List all prediction files
print("Prediction files:")
for p in sorted(PRED_DIR.iterdir()):
    if p.suffix == ".jsonl":
        n_lines = sum(1 for _ in open(p))
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name:50s} {n_lines:>4} records  {size_kb:>8.1f} KB")

## Summary

**Model:** `Qwen/Qwen3-4B-Instruct-2507` — latest Qwen3 generation, 4B dense parameters
(Qwen3 has no 3B dense variant; this is the closest).

**Variants run on val set:**
- **A — Zero-shot:** greedy decoding, no constraints
- **B — Constrained:** same prompt + `outlines` JSON Schema decoding (guarantees valid JSON + correct keys)
- **C — Few-shot (3):** 3 same-category examples from train split, greedy decoding

**Key metrics** (see comparison table above):
- Valid JSON rate: how often the output parses as JSON
- Schema match rate: how often the JSON has exactly the right keys
- Exact match rate: how often prediction == gold (strict, case-sensitive)
- Latency: per-example generation time

**Observations to carry to notebook 08 (evaluation):**
- Check whether Variant B's constrained decoding significantly improves schema adherence
- Check whether few-shot examples help with value extraction accuracy
- Parse failures and schema mismatches need investigation (see section 10)

**Output files (all in `data/WDC-PAVE/predictions/`):**
- `variant_A_zero_shot_val.jsonl`
- `variant_B_constrained_val.jsonl`
- `variant_C_few_shot_3_val.jsonl`

**Next steps:**
- Notebook 06: fine-tuning (Variant D)
- Notebook 07: proprietary API baseline (Variant E)
- Notebook 08: full evaluation with field-level F1, null handling, etc.
- Notebook 09: results comparison and visualization